# International Trade in Goods 5368

## Python set-up

In [1]:
# analytic imports
import pandas as pd
import readabs as ra
from readabs import metacol as mc

# local imports
from abs_helper import get_abs_data
from mgplot import multi_start, seastrend_plot_finalise, fill_between_plot, finalise_plot

# pandas display settings
pd.options.display.max_rows = 999
pd.options.display.max_columns = 999
pd.options.display.max_colwidth = 999

# display charts in this notebook
SHOW = False

## Data capture and plot

In [2]:
abs_dict, meta, source, _ = get_abs_data("5368.0")
plot_times = 0, (4 * -12 -1)
meta.head(10)

Table 5368092 has no 'Index' sheet.
Table 5368018 has no 'Index' sheet.
Table 5368019 has no 'Index' sheet.


8,Data Item Description,Series Type,Series ID,Series Start,Series End,No. Obs.,Unit,Data Type,Freq.,Collection Month,Table,Table Description,Catalogue number
Series ID,,,,,,,,,,,,,
A2718594C,Balance on goods ;,Seasonally Adjusted,A2718594C,1971-07-01 00:00:00,2026-03-01 00:00:00,657,$ Millions,FLOW,Month,1,536801,"GOODS, Summary: Seasonally adjusted and trend estimates, Current prices",5368.0
A2718577A,"Credits, Total goods ;",Seasonally Adjusted,A2718577A,1971-07-01 00:00:00,2026-03-01 00:00:00,657,$ Millions,FLOW,Month,1,536801,"GOODS, Summary: Seasonally adjusted and trend estimates, Current prices",5368.0
A2718580R,"Credits, Rural goods ;",Seasonally Adjusted,A2718580R,1971-07-01 00:00:00,2026-03-01 00:00:00,657,$ Millions,FLOW,Month,1,536801,"GOODS, Summary: Seasonally adjusted and trend estimates, Current prices",5368.0
A2718587F,"Credits, Non-rural goods ;",Seasonally Adjusted,A2718587F,1971-07-01 00:00:00,2026-03-01 00:00:00,657,$ Millions,FLOW,Month,1,536801,"GOODS, Summary: Seasonally adjusted and trend estimates, Current prices",5368.0
A2718599R,"Credits, Net exports of goods under merchanting ;",Seasonally Adjusted,A2718599R,1971-07-01 00:00:00,2026-03-01 00:00:00,657,$ Millions,FLOW,Month,1,536801,"GOODS, Summary: Seasonally adjusted and trend estimates, Current prices",5368.0
A2718602T,"Credits, Non-monetary gold ;",Seasonally Adjusted,A2718602T,1971-07-01 00:00:00,2026-03-01 00:00:00,657,$ Millions,FLOW,Month,1,536801,"GOODS, Summary: Seasonally adjusted and trend estimates, Current prices",5368.0
A2718603V,"Debits, Total goods ;",Seasonally Adjusted,A2718603V,1971-07-01 00:00:00,2026-03-01 00:00:00,657,$ Millions,FLOW,Month,1,536801,"GOODS, Summary: Seasonally adjusted and trend estimates, Current prices",5368.0
A2718605X,"Debits, Consumption goods ;",Seasonally Adjusted,A2718605X,1971-07-01 00:00:00,2026-03-01 00:00:00,657,$ Millions,FLOW,Month,1,536801,"GOODS, Summary: Seasonally adjusted and trend estimates, Current prices",5368.0
A2718612W,"Debits, Capital goods ;",Seasonally Adjusted,A2718612W,1971-07-01 00:00:00,2026-03-01 00:00:00,657,$ Millions,FLOW,Month,1,536801,"GOODS, Summary: Seasonally adjusted and trend estimates, Current prices",5368.0


In [3]:
subset = meta[meta[mc.did].str.contains("Balance on goods")]
seas = subset[subset[mc.stype].str.contains("Seasonally")].index[0]
trend = subset[subset[mc.stype].str.contains("Trend")].index[0]
table = "536801"
units = "$ Millions"
data = abs_dict[table][[seas, trend]].rename(columns={seas: "Seasonally adjusted", trend: "Trend"})
data, units = ra.recalibrate(data, units)
multi_start(
    data,
    function=seastrend_plot_finalise,
    starts=plot_times,
    title="Trade balance on Goods",
    ylabel=units,
    y0=True,
    lfooter="Australia. Current prices. ",
    rfooter=source,
    show=SHOW,
)

## Goods Export Composition by SITC

Quarterly merchandise exports (FOB) by SITC 1-digit, grouped into five narrative buckets (food/ag, crude materials, mineral fuels, manufactures, other incl. gold). Source: ABS 5368.0 table 5368012a.

In [4]:
# --- Quarterly goods export composition (SITC 1-digit) ---
EXP_TABLE = "5368012a"
meta_t = meta[meta[mc.table] == EXP_TABLE].copy()
meta_t["sitc"] = meta_t[mc.did].str.extract(r"^(\d) ", expand=False)
meta_t = meta_t[meta_t["sitc"].notna()]

monthly = pd.concat(
    {row["sitc"]: abs_dict[EXP_TABLE][row[mc.id]] for _, row in meta_t.iterrows()},
    axis=1,
).sort_index(axis=1)

# Monthly $millions -> quarterly sums (flow data)
quarterly = (
    monthly.groupby(monthly.index.asfreq("Q-DEC")).sum(min_count=3).dropna()
)

SITC_GROUPS = {
    "Food, bev., oils (SITC 0+1+4)": ["0", "1", "4"],
    "Crude materials (SITC 2)":      ["2"],
    "Mineral fuels (SITC 3)":        ["3"],
    "Manufactures (SITC 5+6+7+8)":   ["5", "6", "7", "8"],
    "Other incl. gold (SITC 9)":     ["9"],
}
grouped = pd.concat(
    {n: quarterly[codes].sum(axis=1) for n, codes in SITC_GROUPS.items()},
    axis=1,
)

levels_bn = grouped / 1000.0                                    # A$bn
shares    = grouped.div(grouped.sum(axis=1), axis=0) * 100      # %

PALETTE = ["#b8d4a3", "#f4a89c", "#f7d488", "#a3c4e8", "#d6d6d6"]


def _stack(df, *, ylabel, title, tag):
    ax = None
    cumulative = pd.Series(0.0, index=df.index)
    for col, color in zip(df.columns, PALETTE):
        band = pd.DataFrame({"lower": cumulative, "upper": cumulative + df[col]})
        ax = fill_between_plot(band, ax=ax, color=color, alpha=1.0, label=col)
        cumulative = cumulative + df[col]
    finalise_plot(
        ax,
        title=title,
        ylabel=ylabel,
        rfooter=f"{source} {EXP_TABLE}",
        lfooter="Australia. Original. Merchandise exports, FOB, quarterly sums.",
        legend={"loc": "upper left", "fontsize": "x-small", "frameon": False},
        tag=tag,
        show=SHOW,
    )


_stack(shares,    ylabel="Per cent of total goods exports",
       title="Australia goods export composition (SITC, share)", tag="share")
_stack(levels_bn, ylabel="A$ billions per quarter",
       title="Australia goods export composition (SITC, A$bn)",  tag="levels")

## Finished

In [5]:
# watermark
%load_ext watermark
%watermark -u -t -d --iversions --watermark --machine --python --conda

Last updated: 2026-05-23 11:39:35

Python implementation: CPython
Python version       : 3.14.2
IPython version      : 9.13.0

conda environment: n/a

Compiler    : Clang 21.1.4 
OS          : Darwin
Release     : 25.3.0
Machine     : arm64
Processor   : arm
CPU cores   : 14
Architecture: 64bit

mgplot : 0.2.23
pandas : 3.0.3
readabs: 0.1.9

Watermark: 2.6.0

